<a href="https://colab.research.google.com/github/jhenyferssouza/Agente-Orthus/blob/agente-2%2Fmariastella/Agente2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install chromadb groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 88.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 113.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 126.7 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.63b1
    Uninstalling opentelemetry-semantic-conventions-0.63b1:
      Successfully uninstalled opentelemetry-semantic-conventions-0.63b1
  Attempting uninstall: opentelemetry-sdk
    Found existing installation: opentelemetry-sdk 1.42.1
    Uninstalling opentelemetry-sdk-1.42.1:
      S

In [10]:
import json
import chromadb
from groq import Groq
from google.colab import userdata

# ==========================================
# 1. CRIANDO A CLASSE DO AGENTE 2 (CORRIGIDA DEFINITIVAMENTE)
# ==========================================
class Agente2Recomendacao:
    def __init__(self, groq_api_key: str, caminho_banco="./meu_chromadb", nome_colecao="nr17_base"):
        self.chroma_client = chromadb.PersistentClient(path=caminho_banco)
        self.colecao = self.chroma_client.get_collection(name=nome_colecao)
        self.ia_client = Groq(api_key=groq_api_key)

    def _buscar_no_rag(self, tipo_erro: str) -> str:
        """Busca no ChromaDB as regras sobre o erro encontrado."""
        resultados = self.colecao.query(query_texts=[tipo_erro], n_results=1)
        documentos = resultados.get('documents', [])

        if documentos and len(documentos) > 0:
            item = documentos[0]
            if isinstance(item, list) and len(item) > 0:
                return item[0]
            elif isinstance(item, str):
                return item
        return "Diretrizes gerais da NR-17: Manter postura neutra e pescoço alinhado."

    def processar_alerta(self, json_recebido_do_agente_1: str) -> dict:
        """Lê o erro do Agente 1, consulta a NR-17 e gera os alertas/exercícios."""
        dados_alerta = json.loads(json_recebido_do_agente_1)
        issue = dados_alerta.get("issue", "Desvio Postural")
        duration = dados_alerta.get("duration_seconds", 0)
        details = dados_alerta.get("details", "")

        contexto_nr17 = self._buscar_no_rag(issue)

        system_prompt = (
            "Você é um fisioterapeuta do trabalho especialista na lei NR-17. "
            "Gere recomendações baseadas no contexto fornecido. Responda apenas em formato JSON."
        )

        user_prompt = f"""
        [ALERTA DA WEBCAM]
        - Erro detectado: {issue}
        - Tempo na posição: {duration} segundos
        - Ângulo medido: {details}

        [MANUAL TÉCNICO DA NR-17 / CINESIOLOGIA]
        {contexto_nr17}

        Preencha o JSON abaixo estritamente com base nos dados acima (em português):
        {{
            "justificativa": "Por que esse ângulo medido faz mal de acordo com o manual técnico?",
            "correcao": "Instrução curta de como reajustar o corpo ou o monitor agora para voltar ao ângulo seguro.",
            "microexercicio": "Sugestão de um alongamento ativo específico de 15 segundos para a região."
        }}
        """

        # Envia para a IA da Groq processar
        resposta = self.ia_client.chat.completions.create(
            model="qwen/qwen3.8-27b",
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.1
        )

        # FIX DEFINITIVO PARA O SDK DA GROQ:
        # Tratamos se a resposta vier como lista, objeto tradicional ou dicionário bruto
        if isinstance(resposta, list):
            texto_json = resposta[0].message.content
        elif hasattr(resposta, 'choices') and len(resposta.choices) > 0:
            texto_json = resposta.choices[0].message.content
        else:
            # Caso venha como um dicionário puro mapeado por chaves
            texto_json = resposta['choices'][0]['message']['content']

        return json.loads(texto_json)

# ==========================================
# 2. SEU FLUXO DE EXECUÇÃO E TESTE DO ZERO
# ==========================================
print("🚀 Passo 1: Criando banco ChromaDB de teste com regras de ângulos...")
client = chromadb.PersistentClient(path="./meu_chromadb")
colecao = client.get_or_create_collection(name="nr17_base")

try:
    colecao.delete(ids=["doc_cervical"])
except Exception:
    pass

colecao.add(
    documents=[
        "Cinesiologia Cervical: O ângulo seguro da cabeça é de até 15 graus. Ângulos superiores a 30 graus triplicam o peso na coluna, violando os princípios de conforto da NR-17.5.3."
    ],
    ids=["doc_cervical"]
)

print("🔑 Passo 2: Puxando sua chave secreta do cofre do Colab...")
try:
    CHAVE_SECRETA = userdata.get('GROQ_API_KEY')
except Exception:
    CHAVE_SECRETA = None
    print("❌ ERRO: Você esqueceu de colocar a chave no ícone de Chave 🔑 na lateral do Colab!")

if CHAVE_SECRETA:
    agente_postura = Agente2Recomendacao(groq_api_key=CHAVE_SECRETA)

    print("📨 Passo 3: Simulando envio de alerta do Agente 1...")
    json_da_webcam = """
    {
        "event": "POSTURE_PERSISTENCE_ALERT",
        "issue": "Inclinação Cervical",
        "duration_seconds": 15,
        "details": "Pescoço inclinado em 35 graus (Limite seguro é 15 graus)."
    }
    """

    print("🤖 Passo 4: Agente 2 trabalhando...")
    resultado = agente_postura.processar_alerta(json_da_webcam)

    print("\n🎉 CONCLUÍDO COM SUCESSO! VEJA O CARD GERADO PARA O SEU DASHBOARD:")
    print(json.dumps(resultado, indent=4, ensure_ascii=False))



🚀 Passo 1: Criando banco ChromaDB de teste com regras de ângulos...
🔑 Passo 2: Puxando sua chave secreta do cofre do Colab...
📨 Passo 3: Simulando envio de alerta do Agente 1...
🤖 Passo 4: Agente 2 trabalhando...

🎉 CONCLUÍDO COM SUCESSO! VEJA O CARD GERADO PARA O SEU DASHBOARD:
{
    "justificativa": "O ângulo de 35 graus excede o limite seguro de 15 graus e ultrapassa a marca de 30 graus, o que triplica a carga sobre a coluna cervical, violando os princípios de conforto e ergonomia da NR-17.5.3.",
    "correcao": "Eleve o monitor para que o topo da tela fique na altura dos olhos e recue a cadeira para alinhar a coluna, mantendo o pescoço neutro.",
    "microexercicio": "Incline a cabeça lateralmente para a direita, puxando suavemente com a mão direita por 15 segundos, e repita do lado esquerdo."
}


In [12]:
%%writefile base_ergonomica.md
# BASE DE CONHECIMENTO ERGONÔMICA - DIRETRIZES NR-17 E CINESIOLOGIA

## 1. Região Cervical (Pescoço e Cabeça)
- **Diretriz NR-17**: O monitor de vídeo deve estar posicionado na altura dos olhos e com inclinação adequada, permitindo uma posição neutra da cabeça e pescoço para evitar fadiga visual e muscular.
- **Ângulo Neutro (Seguro)**: Inclinação para frente (flexão) de até 15 graus em relação à linha vertical do tronco.
- **Zona de Risco**: Inclinação cervical superior a 30 graus. A partir deste ângulo, o peso da cabeça sobre os discos da coluna triplica, tensionando severamente o músculo trapézio.
- **Correção**: Elevar o monitor (utilizando um suporte) para alinhar o topo da tela com a linha dos olhos.
- **Microexercício**: Entrelace os dedos atrás da nuca e empurre a cabeça suavemente para trás contra as mãos por 15 segundos.

## 2. Região Lombar e Tronco (Costas)
- **Diretriz NR-17**: Os assentos devem possuir encosto com formato levemente adaptado à curvatura lombar e ajuste de inclinação para fornecer suporte ativo à região dorsal durante toda a jornada.
- **Ângulo Neutro (Seguro)**: Tronco alinhado na vertical ou levemente inclinado para trás, mantendo um ângulo entre 90 e 110 graus em relação às coxas.
- **Zona de Risco**: Tronco inclinado ou curvado para frente em um ângulo maior que 20 graus (postura corcunda/cifose). Isso gera compressão severa nos discos vertebrais lombares.
- **Correção**: Sentar aproximando o quadril até o fundo da cadeira e apoiar totalmente as costas no encosto.
- **Microexercício**: Fique de pé, coloque as mãos na base da lombar e incline suavemente o tronco para trás por 10 segundos para aliviar a compressão.

## 3. Ombros e Cintura Escapular
- **Diretriz NR-17**: Os postos de trabalho devem permitir que os braços fiquem posicionados junto ao corpo e os ombros relaxados, evitando tensões musculares estáticas na região do pescoço e conexões dos braços.
- **Ângulo Neutro (Seguro)**: Ombros nivelados (simétricos), alinhados lateralmente com as orelhas, sem elevação ou projeção anterior.
- **Zona de Risco**: Ombros caídos para frente (protrusão) ou tensionados/erguidos em direção às orelhas por mais de 5 centímetros de desnível.
- **Correção**: Puxar os ombros ligeiramente para trás e para baixo, relaxando os braços sobre os apoios.
- **Microexercício**: Realize um movimento circular com os dois ombros para trás de forma lenta por 15 segundos.

## 4. Membros Superiores (Braços, Cotovelos e Punhos)
- **Diretriz NR-17**: O teclado e o mouse devem estar posicionados de forma que o usuário não precise esticar os braços ou dobrar os punhos de forma prejudicial para digitar.
- **Ângulo Neutro (Seguro)**: Cotovelos dobrados bem próximos a um ângulo de 90 graus. Punhos retos e alinhados com o antebraço (ângulo neutro de 0 a 10 graus de flexão/extensão).
- **Zona de Risco**: Cotovelos muito abertos ou punhos dobrados para cima ou para os lados (desvio ulnar/radial) durante a digitação.
- **Correção**: Ajustar a altura da cadeira ou da mesa para que os antebraços fiquem paralelos ao chão ao digitar.
- **Microexercício**: Estique o braço para a frente com a palma da mão voltada para cima e puxe os dedos para baixo com a outra mão por 15 segundos.


Writing base_ergonomica.md


In [13]:
import chromadb

# 1. Conecta ao banco existente no Colab
client = chromadb.PersistentClient(path="./meu_chromadb")

# Tenta recriar a coleção do zero para limpar as regras velhas e carregar as novas
try:
    client.delete_collection(name="nr17_base")
except Exception:
    pass

colecao = client.get_or_create_collection(name="nr17_base")

# 2. Criamos os blocos (chunks) baseados no nosso arquivo Markdown
novos_documentos = [
    "Cervical (Pescoço/Cabeça) - NR-17.5.3: Monitor na altura dos olhos para posição neutra. Ângulo seguro: inclinação para frente de até 15°. Risco: acima de 30° triplica o peso na coluna. Correção: erguer monitor. Exercício: empurrar cabeça para trás contra as mãos por 15s.",
    "Lombar e Tronco (Costas) - NR-17: Assento com suporte lombar ativo. Ângulo seguro: tronco entre 90° e 110° com as coxas. Risco: tronco inclinado para frente acima de 20° (corcunda). Correção: encostar quadril no fundo da cadeira. Exercício: de pé, inclinar levemente para trás por 10s.",
    "Ombros e Escápulas - NR-17: Braços junto ao corpo e ombros relaxados. Ângulo seguro: ombros nivelados e alinhados com as orelhas. Risco: ombros caídos para frente ou erguidos. Correção: puxar ombros para trás e para baixo. Exercício: rotação de ombros para trás por 15s.",
    "Membros Superiores (Punhos/Braços) - NR-17: Teclado/mouse ao alcance sem dobrar punhos. Ângulo seguro: cotovelos a 90° e punhos retos (0° a 10°). Risco: punhos dobrados ou cotovelos muito abertos. Correção: ajustar altura do assento. Exercício: alongamento de punho puxando os dedos para baixo por 15s."
]

novos_ids = ["doc_cervical", "doc_lombar", "doc_ombros", "doc_punhos"]

# 3. Adiciona todos os blocos no ChromaDB
colecao.add(documents=novos_documentos, ids=novos_ids)
print("📚 Nova Base Ergonômica Multi-Articulação carregada com sucesso no ChromaDB!")


📚 Nova Base Ergonômica Multi-Articulação carregada com sucesso no ChromaDB!


In [15]:
import json
import chromadb
from google.colab import userdata

# 1. Conecta ao banco e atualiza os documentos das novas articulações
client = chromadb.PersistentClient(path="./meu_chromadb")

try:
    client.delete_collection(name="nr17_base")
except Exception:
    pass

colecao = client.get_or_create_collection(name="nr17_base")

novos_documentos = [
    "Cervical (Pescoço/Cabeça) - NR-17.5.3: Monitor na altura dos olhos para posição neutra. Ângulo seguro: inclinação para frente de até 15°. Risco: acima de 30° triplica o peso na coluna. Correção: erguer monitor. Exercício: empurrar cabeça para trás contra as mãos por 15s.",
    "Lombar e Tronco (Costas) - NR-17: Assento com suporte lombar ativo. Ângulo seguro: tronco entre 90° e 110° com as coxas. Risco: tronco inclinado para frente acima de 20° (corcunda). Correção: encostar quadril no fundo da cadeira. Exercício: de pé, inclinar levemente para trás por 10s.",
    "Ombros e Escápulas - NR-17: Braços junto ao corpo e ombros relaxados. Ângulo seguro: ombros nivelados e alinhados com as orelhas. Risco: ombros caídos para frente ou erguidos. Correção: puxar ombros para trás e para baixo. Exercício: rotação de ombros para trás por 15s.",
    "Membros Superiores (Punhos/Braços) - NR-17: Teclado/mouse ao alcance sem dobrar punhos. Ângulo seguro: cotovelos a 90° e punhos retos (0° a 10°). Risco: punhos dobrados ou cotovelos muito abertos. Correção: ajustar altura do assento. Exercício: alongamento de punho puxando os dedos para baixo por 15s."
]
novos_ids = ["doc_cervical", "doc_lombar", "doc_ombros", "doc_punhos"]
colecao.add(documents=novos_documentos, ids=novos_ids)
print("📚 1. Nova Base Ergonômica carregada no ChromaDB!")

# 2. Puxa a chave de segurança e REINICIALIZA o agente para atualizar o vínculo com o banco
CHAVE_SECRETA = userdata.get('GROQ_API_KEY')
agente_postura = Agente2Recomendacao(groq_api_key=CHAVE_SECRETA, caminho_banco="./meu_chromadb", nome_colecao="nr17_base")
print("🤖 2. Agente 2 reinicializado e sincronizado com o novo banco!")

# 3. Executa o teste com o JSON de múltiplos desvios simultâneos
json_multi_erro = """
{
    "event": "POSTURE_PERSISTENCE_ALERT",
    "issue": "Inclinação Cervical e Desvio Lombar corcunda",
    "duration_seconds": 25,
    "details": "Pescoço inclinado para frente a 34 graus e tronco curvado (corcunda) a 28 graus."
}
"""

print("\n🚀 3. Agente 2 analisando múltiplos ângulos...")
resultado_multiplo = agente_postura.processar_alerta(json_multi_erro)

print("\n🎉 CONCLUÍDO COM SUCESSO! SEU CARD MULTI-ARTICULAÇÃO:")
print(json.dumps(resultado_multiplo, indent=4, ensure_ascii=False))


📚 1. Nova Base Ergonômica carregada no ChromaDB!
🤖 2. Agente 2 reinicializado e sincronizado com o novo banco!

🚀 3. Agente 2 analisando múltiplos ângulos...

🎉 CONCLUÍDO COM SUCESSO! SEU CARD MULTI-ARTICULAÇÃO:
{
    "justificativa": "A inclinação cervical de 34° excede o limite seguro de 15° e ultrapassa o limiar de 30°, o que, conforme o manual, triplica o peso suportado pela coluna cervical, aumentando significativamente o risco de lesões.",
    "correcao": "Erga o monitor para que a tela fique na altura dos olhos, garantindo a posição neutra da cabeça e reduzindo a curvatura lombar.",
    "microexercicio": "Empurre a cabeça para trás contra as mãos por 15 segundos."
}


# 📑 CONTRATO DE INTEGRAÇÃO: AGENTE 1 ➔ AGENTE 2

Este documento serve como o "manual de encaixe" para juntar o **Agente 1** (Visão Computacional + Janela Móvel) com o **Agente 2** (RAG + Recomendações). Como o sistema é desacoplado, a comunicação depende exclusivamente de um formato de dados unificado.

---

## 🤝 O Acordo de Encaixe (Contrato JSON)

Para que o **Agente 2** consiga consultar a base ergonômica da NR-17 no ChromaDB e gerar as respostas com IA, o **Agente 1** precisa obrigatoriamente disparar uma string ou dicionário JSON exatamente com a estrutura de chaves abaixo quando a janela limite de erro estourar:

```json
{
    "event": "POSTURE_PERSISTENCE_ALERT",
    "issue": "Nome do Erro Postural",
    "duration_seconds": 15,
    "details": "Texto descritivo com as métricas e ângulos medidos pelo YOLO."
}
```

### 🔍 Dicionário de Chaves Obrigatórias:
*   `event`: String estática identificando o tipo de evento (deve ser sempre `"POSTURE_PERSISTENCE_ALERT"`).
*   `issue`: O termo ou articulação onde o erro ocorreu (Ex: `"Inclinação Cervical"`, `"Desvio Lombar"`, `"Ombros Assimétricos"`). É essa string que o Agente 2 usa para buscar informações no ChromaDB.
*   `duration_seconds`: O tempo em segundos que o usuário permaneceu naquela posição inadequada antes do alerta disparar.
*   `details`: Informações métricas brutas geradas pelos cálculos matemáticos dos keypoints (Ex: `"Pescoço inclinado a 35 graus (Limite seguro é 15 graus)."`).

---

## ⚙️ Como os Códigos se Conectam (Exemplo no Streamlit/Main)

No script principal da aplicação (na integração dentro do Dashboard), o fluxo de dados frame-a-frame da webcam seguirá o seguinte fluxo em Python:

```python
# 1. O Agente 1 roda continuamente processando os frames da webcam
erro_detectado, json_gerado_pelo_agente_1 = agente_1.analisar_frame_webcam(frame)

# 2. Se a lógica de Janela Móvel do Agente 1 estourar o tempo limite (X segundos):
if erro_detectado:
    
    # 3. O JSON do Agente 1 é injetado diretamente na função do SEU Agente 2
    card_recomendacao = agente_postura.processar_alerta(json_gerado_pelo_agente_1)
    
    # 4. O Dashboard do Streamlit renderiza as chaves do dicionário gerado pela IA
    st.error("🚨 ALERTA ERGONÔMICO DETECTADO!")
    st.markdown(f"**Por que faz mal:** {card_recomendacao['justificativa']}")
    st.markdown(f"**Como corrigir agora:** {card_recomendacao['correcao']}")
    st.success(f"**🧘 Microexercício:** {card_recomendacao['microexercicio']}")
```

---

## 🎯 Formato de Saída Garantido pelo Agente 2

Após receber o JSON do Agente 1 e consultar a base da NR-17 via RAG, o Agente 2 devolve para a tela um dicionário Python limpo com três chaves estruturadas. Isso garante que a interface do Streamlit nunca quebre ao renderizar os dados:

```json
{
    "justificativa": "Explicação concisa e clara baseada estritamente na NR-17/Cinesiologia sobre os riscos do ângulo medido.",
    "correcao": "Instruções acionáveis e curtas de como reajustar o monitor, a cadeira ou o próprio corpo imediatamente.",
    "microexercicio": "Sugestão de um alongamento rápido ou mobilidade ativa de até 15 segundos voltado para aliviar a articulação afetada."
}
```


# 🛠️ Documentação do Projeto - Agente 2: Geração Explicável de Recomendações

Este módulo contém a arquitetura, o código de produção e o guia de integração do **Agente 2**, um agente cognitivo de IA que atua integrado a um banco de dados vetorial (RAG) para mitigar riscos ergonômicos em tempo real. O sistema foi desenvolvido sob diretrizes estritas da **NR-17** e conceitos biomecânicos de **Cinesiologia**.

---

## 🧠 1. Arquitetura de Funcionamento

O Agente 2 opera de forma totalmente desacoplada do hardware de captura, seguindo o pipeline abaixo:

1. **Ingestão (Input)**: O agente intercepta um payload JSON estruturado disparado pelo Agente 1 (Visão Computacional) assim que uma postura nociva atinge a janela móvel de tempo limite.
2. **Busca Vetorial (RAG)**: O termo de desvio (`issue`) é extraído e utilizado em uma busca semântica no banco de dados vetorial **ChromaDB**.
3. **Recuperação de Contexto**: O ChromaDB retorna as regras de conforto da NR-17 e os ângulos de risco cinesiologistas equivalentes à articulação afetada.
4. **Prompt Engineering & Grounding**: O contexto do manual técnico é injetado junto com as métricas em tempo real e regras estritas de não-alucinação em um prompt instrucional.
5. **Inferência Confiável**: A requisição é processada pela infraestrutura de ultra-baixa latência da **Groq API** utilizando o modelo de produção **`qwen/qwen3.8-27b`** (ou `openai/gpt-oss-20b`). A temperatura é fixada em `0.1` para garantir respostas 100% ancoradas na base jurídica e científica.
6. **Entrega Estruturada (Output)**: O agente devolve um objeto JSON tratado contendo Justificativa, Correção e Microexercício pronto para renderização visual no dashboard.

---

## 🤝 2. Contrato de Integração (Conexão com o Agente 1)

Para garantir que a integração de ponta a ponta ocorra sem conflitos de software, o **Agente 1** deve obrigatoriamente construir e despachar um objeto JSON que respeite a seguinte estrutura de chaves:

```json
{
    "event": "POSTURE_PERSISTENCE_ALERT",
    "issue": "Inclinação Cervical",
    "duration_seconds": 15,
    "details": "Pescoço inclinado em 35 graus (Limite seguro é 15 graus)."
}
```

### 📋 Mapeamento de Chaves Obrigatórias:
*   `event`: String estática identificadora (`"POSTURE_PERSISTENCE_ALERT"`).
*   `issue`: A articulação ou desvio mapeado (Ex: `"Inclinação Cervical"`, `"Desvio Lombar"`, `"Ombros Assimétricos"`, `"Desvio de Punho"`). *Esta string dita a chave de busca semântica no ChromaDB.*
*   `duration_seconds`: O tempo em segundos de persistência da postura incorreta.
*   `details`: Informações métricas e angulares brutas geradas pelos cálculos matemáticos sobre as coordenadas da pose.

---

## 💻 3. Código de Produção do Agente 2

Abaixo encontra-se a implementação completa e blindada da classe do agente, projetada para tratar as respostas da API da Groq de forma robusta, independente das variações estruturais de formato do SDK:

```python
import json
import chromadb
from groq import Groq
from google.colab import userdata

class Agente2Recomendacao:
    def __init__(self, groq_api_key: str, caminho_banco="./meu_chromadb", nome_colecao="nr17_base"):
        # Conecta ao banco de dados onde ficam os textos da NR-17
        self.chroma_client = chromadb.PersistentClient(path=caminho_banco)
        self.colecao = self.chroma_client.get_collection(name=nome_colecao)
        # Conecta com a IA da Groq
        self.ia_client = Groq(api_key=groq_api_key)

    def _buscar_no_rag(self, tipo_erro: str) -> str:
        """Busca no ChromaDB as regras sobre o erro encontrado."""
        resultados = self.colecao.query(query_texts=[tipo_erro], n_results=1)
        documentos = resultados.get('documents', [])
        
        if documentos and len(documentos) > 0:
            item = documentos[0]
            if isinstance(item, list) and len(item) > 0:
                return item[0]
            elif isinstance(item, str):
                return item
        return "Diretrizes gerais da NR-17: Manter postura neutra e pescoço alinhado."

    def processar_alerta(self, json_recebido_do_agente_1: str) -> dict:
        """Lê o erro do Agente 1, consulta a NR-17 e gera os alertas/exercícios."""
        dados_alerta = json.loads(json_recebido_do_agente_1)
        issue = dados_alerta.get("issue", "Desvio Postural")
        duration = dados_alerta.get("duration_seconds", 0)
        details = dados_alerta.get("details", "")

        # Busca o manual de regras no ChromaDB (RAG)
        contexto_nr17 = self._buscar_no_rag(issue)

        system_prompt = (
            "Você é um fisioterapeuta do trabalho especialista na lei NR-17. "
            "Gere recomendações baseadas no contexto fornecido. Responda apenas em formato JSON."
        )

        user_prompt = f"""
        [ALERTA DA WEBCAM]
        - Erro detectado: {issue}
        - Tempo na posição: {duration} segundos
        - Ângulo medido: {details}

        [MANUAL TÉCNICO DA NR-17 / CINESIOLOGIA]
        {contexto_nr17}

        Preencha o JSON abaixo estritamente com base nos dados acima (em português):
        {{
            "justificativa": "Por que esse ângulo medido faz mal de acordo com o manual técnico?",
            "correcao": "Instrução curta de como reajustar o corpo ou o monitor agora para voltar ao ângulo seguro.",
            "microexercicio": "Sugestão de um alongamento ativo específico de 15 segundos para a região."
        }}
        """

        # Envia para a IA da Groq processar com o modelo estável de produção
        resposta = self.ia_client.chat.completions.create(
            model="qwen/qwen3.8-27b",
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.1
        )
        
        # Tratamento seguro para extração do conteúdo de texto do JSON da IA
        if isinstance(resposta, list):
            texto_json = resposta[0].message.content
        elif hasattr(resposta, 'choices') and len(resposta.choices) > 0:
            texto_json = resposta.choices[0].message.content
        else:
            texto_json = resposta['choices'][0]['message']['content']
            
        return json.loads(texto_json)
```

---

## 🔒 4. Camada de Segurança e Ocultação de Chaves

O sistema proíbe a exposição de chaves privadas (`API Keys`) de forma explícita no código para evitar vazamentos em repositórios públicos como o GitHub.

### No Google Colab (Ambiente de Testes)
A credencial é injetada via painel **Secrets** (ícone de Chave 🔑 na barra lateral esquerda), sob o rótulo identificador de `GROQ_API_KEY`. O resgate do hash seguro ocorre via API do Google:
```python
from google.colab import userdata
CHAVE_SECRETA = userdata.get('GROQ_API_KEY')
```

### No Streamlit Local (Ambiente de Produção)
As chaves de desenvolvimento devem ser inseridas no arquivo oculto local `.streamlit/secrets.toml`:
```toml
GROQ_API_KEY = "gsk_seu_token_aqui"
```
E acessadas dinamicamente sem exposição via código com `st.secrets["GROQ_API_KEY"]`.

---

## 🎯 5. Critérios de Aceite Homologados

*   **Explicabilidade Técnica**: Geração automática de justificativas com base direta na NR-17 e dados angulares sem alucinações técnicas.
*   **Saída Estruturada**: Resposta nativa em formato JSON garantindo conformidade com a camada de visualização no Dashboard.
*   **Análise Multi-Articulação**: Pronto para processar desvios de múltiplos eixos biomecânicos e articulações simultâneas.
